In [2]:
import pandas as pd
import numpy as np
import json
pd.set_option('display.max_columns', 500)

YB_TOKENS = [
    'eth_usr_usdc', 'eth_wsteth_usdc', 'eth_rlp_usdc',
    'eth_usd0++_usdc', 'eth_fxsave_usdc', 'eth_mapollo_usdc',
    'eth_wsrusd_usdc', 'eth_syrupusdc_pyusd', 'eth_susde_pyusd',
    'eth_stcusd_usdc', 'eth_usde_dai', 'eth_mhyper_usdc', 'eth_syrupusdc_usdc',
    'eth_wstusr_usdc','eth_slvlusd_usdc','eth_csusdl_usdc', 'eth_mF-ONE_usdc', 'eth_reusd_usdc',
    'eth_siusd_usdc', 'eth_sdeusd_usdc'
]



# MARKET = "base_cbbtc_usdc_full"
# MARKET = "eth_cbbtc_usdc"
MARKET = "eth_susde_pyusd"
EVENTS_PATH = "/Users/yegortrussov/Documents/ml/lending_protocols/dataset_collection/data/markets_enriched"
HOURLY_MARKET_PATH = "/Users/yegortrussov/Documents/ml/lending_protocols/dataset_collection/data/markets_hourly_data"
POSITIONS_PATH = "/Users/yegortrussov/Documents/ml/lending_protocols/dataset_collection/data/users_positions/crypto_tokens_positions.csv"

df = pd.read_csv(f"{EVENTS_PATH}/{MARKET}.csv")
market_df = pd.read_csv(f"{HOURLY_MARKET_PATH}/{MARKET}.csv")

with open("/Users/yegortrussov/Documents/ml/lending_protocols/dataset_collection/data/common/markets_meta.json", 'r') as f:
    markets_meta = json.load(f)
with open("/Users/yegortrussov/Documents/ml/lending_protocols/dataset_collection/data/common/assets_meta.json", 'r') as f:
    assets_meta = json.load(f)
market_addr = df["market_address"].unique()[0]
market_meta = markets_meta[market_addr]
asset_meta = assets_meta[market_meta["collateral_asset_address"]]

df.head(2)

,hash,type,timestamp,user_address,assets,assets_usd,liquidated_assets,liquidated_assets_usd,market,datetime,market_address,total_supply_before,total_borrow_before,total_supply_after,total_borrow_after,utilization_before,utilization_after,tx_actions,borrow_rate_before,supply_rate_before,borrow_rate_after,supply_rate_after,collateral_price,loan_asset_price,collateral_before,collateral_value_before,debt_before,supply_before,ltv_before,collateral_after,collateral_value_after,debt_after,supply_after,ltv_after,health_factor_before,health_factor_after,event_type,vault_flg,volatility_6h,drawdown_6h,trend_6h,volatility_24h,drawdown_24h,trend_24h,event_sequence_type,collateral_asset_symbol,loan_asset_symbol
0,0xda3a5b0ec6ecb57c22993164b41c1a4dc67723197869...,MarketSupply,1765732379,0x000000000000000000000000000000000000dEaD,1000,0.001000,0,0,eth_susde_pyusd,2025-12-14 17:12:59,0x90ef0c5a0dc7c4de4ad4585002d44e9d411d212d2f62...,0.000,0.0,0.001,0.0,0.0,0.0,1,0.009999,0.0,0.009999,0.0,1.211553,0.999609,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.001,0.0,0.0,0.0,loan_position_supply,False,0.000535,-0.001066,-0.000216,0.000388,-0.001066,0.000925,loan_position_supply,sUSDe,PYUSD
1,0xa0096177498ecbf80b66f3452e391787e0f9a8129bda...,MarketSupplyCollateral,1765732391,0x13DE0cEE0B83562CBfD46682e10FfA4E3c5090e1,1200000000000000000,1.453864,0,0,eth_susde_pyusd,2025-12-14 17:13:11,0x90ef0c5a0dc7c4de4ad4585002d44e9d411d212d2f62...,0.001,0.0,0.001,0.0,0.0,0.0,1,0.009999,0.0,0.009999,0.0,1.211553,0.999609,0.0,0.0,0.0,0.0,0.0,1.2,1.453864,0.0,0.000,0.0,0.0,0.0,collateral_add,False,0.000535,-0.001066,-0.000216,0.000388,-0.001066,0.000925,collateral_add,sUSDe,PYUSD


In [4]:
%load_ext autoreload
%autoreload 2
from utils import (
    select_users_by_period,
    create_hourly_user_dataset,
)
from visualization_utils import (
    plot_market_features,
)

In [5]:
def hourly_to_daily(hourly_df):
    df = hourly_df.copy()
    df['date'] = pd.to_datetime(df['datetime']).dt.date
    
    # Helper for weighted average
    def weighted_mean(x, weight_col):
        if (x[weight_col] == 0).all():
            return np.nan
        return np.average(x['borrow_rate'], weights=x[weight_col])
    
    daily = df.groupby('date').agg(
        total_supply=('total_supply', 'last'),
        total_borrow=('total_borrow', 'last'),
        utilization=('utilization', 'mean'),
        borrow_rate=('borrow_rate', 'mean'),
        supply_rate=('supply_rate', 'mean'),
        collateral_price=('collateral_price', 'last'),
        loan_asset_price=('loan_asset_price', 'last'),
        asset_price=('asset_price', 'last')
    ).reset_index()
    daily['datetime'] = pd.to_datetime(daily['date'])
    return daily
daily_market_features = hourly_to_daily(market_df)

In [3]:
plot_market_features(
    daily_market_features,
    fields=["total_borrow", "borrow_rate"]
)

NameError: name 'plot_market_features' is not defined